# 06 - Final Report Model Lock

This notebook is the single source of truth for the **final models and outputs used in the report**. It is intentionally separate from `03_model_training.ipynb` and `04_interpretability.ipynb`, which remain exploratory and are left unchanged.

## What this notebook locks

- the exact **feature sets** used in the final report
- the exact **model parameters** used for the three reported systems
- the final **Table 1** metrics
- the final **stage-wise summary** used in the report narrative and figure set
- the final **overall and stage-wise confusion matrices**

## Locked design decisions

1. Remove `abs_elo_gap` from the team-based feature set because it is perfectly collinear with `elo_gap` in the favorite/underdog framing.
2. Use the same adjusted team-only feature set for both the baseline and team-history models.
3. Define the final player-enriched model as the adjusted team set **plus favorite/underdog player and squad columns**, but **exclude derived `diff_*` player features**.
4. Use the exact holdout years from the report: **2018 and 2022**.

If this notebook disagrees with earlier exploratory notebook prose or outputs, **this notebook is authoritative for the final report**.


In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import label_binarize

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

NOTEBOOK_DIR = Path.cwd()
DATA = NOTEBOOK_DIR / 'data'
DB_PATH = NOTEBOOK_DIR / '../../../../data/raw/worldcup/data-sqlite/worldcup.db'
LOCK_DIR = DATA / 'final_report_lock'
SEED = 42
HOLDOUT_YEARS = [2018, 2022]
CLASS_LABELS = ['fav_loses', 'draw', 'fav_wins']

print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('DB_PATH      =', DB_PATH)
print('DATA         =', DATA)


## A. Rebuild the final favorite/underdog modeling table

This section mirrors the `03` pipeline closely enough to preserve the same data framing, but keeps the code here self-contained so the final report can be reproduced from a single notebook.


In [ ]:
feat_team = pd.read_parquet(DATA / 'features_team.parquet')
feat_ps = pd.read_parquet(DATA / 'features_player_squad.parquet')

con = sqlite3.connect(DB_PATH)
def _sql(q):
    return pd.read_sql_query(q, con)

matches = _sql('SELECT * FROM matches')
tournaments = _sql('SELECT tournament_id, tournament_name, year, host_country FROM tournaments')
teams_db = _sql('SELECT team_id, confederation_id, team_name FROM teams')
con.close()

matches['match_dt'] = pd.to_datetime(matches['match_date'], unit='D', origin='unix', errors='coerce')
men_ids = tournaments.loc[
    ~tournaments['tournament_name'].str.contains('Women', case=False, na=False),
    'tournament_id',
].unique()
matches = matches[matches['tournament_id'].isin(men_ids)].copy()
matches = matches.merge(
    tournaments[['tournament_id', 'year', 'host_country']],
    on='tournament_id',
    how='left',
)


def _norm(s):
    return s.astype(str).str.strip().str.lower().str.replace(' ', '', regex=False)


# Rest-day features within each tournament.
sched = matches[['match_id', 'tournament_id', 'home_team_id', 'away_team_id', 'match_dt', 'year']].copy()
long = pd.concat(
    [
        sched.rename(columns={'home_team_id': 'team_id', 'away_team_id': 'opp_id'}),
        sched.rename(columns={'away_team_id': 'team_id', 'home_team_id': 'opp_id'}),
    ],
    ignore_index=True,
).sort_values(['tournament_id', 'team_id', 'match_dt', 'match_id'])
long['prev_dt'] = long.groupby(['tournament_id', 'team_id'])['match_dt'].shift(1)
long['rest_days'] = (long['match_dt'] - long['prev_dt']).dt.days
rest_home = long.rename(columns={'team_id': 'home_team_id', 'rest_days': 'home_rest_days'})
rest_away = long.rename(columns={'team_id': 'away_team_id', 'rest_days': 'away_rest_days'})
matches = matches.merge(
    rest_home[['match_id', 'home_team_id', 'home_rest_days']],
    on=['match_id', 'home_team_id'],
    how='left',
)
matches = matches.merge(
    rest_away[['match_id', 'away_team_id', 'away_rest_days']],
    on=['match_id', 'away_team_id'],
    how='left',
)

# Host flags.
team_name_map = teams_db.set_index('team_id')['team_name']
matches['home_name'] = matches['home_team_id'].map(team_name_map)
matches['away_name'] = matches['away_team_id'].map(team_name_map)
matches['feat_home_is_host'] = (_norm(matches['home_name']) == _norm(matches['host_country'].fillna(''))).astype(int)
matches['feat_away_is_host'] = (_norm(matches['away_name']) == _norm(matches['host_country'].fillna(''))).astype(int)

# Team feature joins in home/away form.
ft_home = feat_team.add_prefix('home_').rename(columns={'home_tournament_id': 'tournament_id', 'home_team_id': 'home_team_id'})
ft_away = feat_team.add_prefix('away_').rename(columns={'away_tournament_id': 'tournament_id', 'away_team_id': 'away_team_id'})
df_all = matches.merge(ft_home, on=['tournament_id', 'home_team_id'], how='left')
df_all = df_all.merge(ft_away, on=['tournament_id', 'away_team_id'], how='left')
df_all['feat_same_confederation'] = (df_all['home_confederation_id'] == df_all['away_confederation_id']).astype(int)

# Favorite / underdog framing.
home_elo = df_all['home_elo_rating'].fillna(0)
away_elo = df_all['away_elo_rating'].fillna(0)
home_is_fav = home_elo >= away_elo
team_cols = [
    c for c in feat_team.columns
    if c not in ('tournament_id', 'team_id', 'year', 'team_name', 'team_code', 'confederation_id')
]

fav_rows = []
for i, (_, row) in enumerate(df_all.iterrows()):
    fav_side = 'home' if home_is_fav.iloc[i] else 'away'
    und_side = 'away' if fav_side == 'home' else 'home'
    r = {
        'match_id': row['match_id'],
        'tournament_id': row['tournament_id'],
        'year': row['year'],
        'group_stage': row.get('group_stage', np.nan),
        'knockout_stage': row.get('knockout_stage', np.nan),
        'feat_home_is_host': row['feat_home_is_host'],
        'feat_away_is_host': row['feat_away_is_host'],
        'feat_same_confederation': row['feat_same_confederation'],
        'fav_rest_days': row.get(f'{fav_side}_rest_days', np.nan),
        'und_rest_days': row.get(f'{und_side}_rest_days', np.nan),
        'fav_is_home': int(fav_side == 'home'),
        'fav_team_id': row[f'{fav_side}_team_id'],
        'und_team_id': row[f'{und_side}_team_id'],
    }
    for col in team_cols:
        r[f'fav_{col}'] = row.get(f'{fav_side}_{col}', np.nan)
        r[f'und_{col}'] = row.get(f'{und_side}_{col}', np.nan)

    fav_win = row.get(f'{fav_side}_team_win', np.nan)
    draw = row.get('draw', np.nan)
    if pd.isna(fav_win):
        r['y'] = np.nan
    elif draw == 1:
        r['y'] = 1
    elif fav_win == 1:
        r['y'] = 2
    else:
        r['y'] = 0
    fav_rows.append(r)

df_fav = pd.DataFrame(fav_rows).dropna(subset=['y']).copy()
df_fav['y'] = df_fav['y'].astype(int)

# Player / squad joins in favorite / underdog form.
ps_cols = [c for c in feat_ps.columns if c not in ('tournament_id', 'team_id', 'national_team', 'year')]
ps_fav = feat_ps.rename(columns={c: f'fav_{c}' for c in ps_cols}).rename(columns={'team_id': 'fav_team_id'})
ps_und = feat_ps.rename(columns={c: f'und_{c}' for c in ps_cols}).rename(columns={'team_id': 'und_team_id'})

df_fav = df_fav.merge(
    ps_fav[['tournament_id', 'fav_team_id'] + [f'fav_{c}' for c in ps_cols]],
    on=['tournament_id', 'fav_team_id'],
    how='left',
)
df_fav = df_fav.merge(
    ps_und[['tournament_id', 'und_team_id'] + [f'und_{c}' for c in ps_cols]],
    on=['tournament_id', 'und_team_id'],
    how='left',
)

# Derived features.
df_fav['elo_gap'] = df_fav['fav_elo_rating'] - df_fav['und_elo_rating']
df_fav['abs_elo_gap'] = df_fav['elo_gap'].abs()
df_fav['hist_win_rate_diff'] = df_fav['fav_hist_win_rate_shrunk'] - df_fav['und_hist_win_rate_shrunk']
df_fav['hist_draw_rate_diff'] = df_fav['fav_hist_draw_rate_shrunk'] - df_fav['und_hist_draw_rate_shrunk']
df_fav['rest_days_diff'] = df_fav['fav_rest_days'] - df_fav['und_rest_days']
for col in ps_cols:
    df_fav[f'diff_{col}'] = df_fav[f'fav_{col}'] - df_fav[f'und_{col}']

print('df_fav shape:', df_fav.shape)
print('Label distribution:')
display(df_fav['y'].value_counts().sort_index().rename(index={0: 'fav_loses', 1: 'draw', 2: 'fav_wins'}).to_frame('count'))
print('Available years:', sorted(df_fav['year'].dropna().astype(int).unique()))


## B. Define the final locked feature sets

The most important distinction from the exploratory work is that the **final player-enriched model does not use `diff_*` player features**. The exact report-aligned feature sets are defined here.


In [ ]:
TEAM_HIST_COLS = [
    'fav_hist_win_rate_shrunk', 'und_hist_win_rate_shrunk',
    'fav_hist_draw_rate_shrunk', 'und_hist_draw_rate_shrunk',
    'fav_hist_ko_win_rate_shrunk', 'und_hist_ko_win_rate_shrunk',
    'fav_hist_goal_diff_per_match', 'und_hist_goal_diff_per_match',
    'fav_hist_frac_ko', 'und_hist_frac_ko',
    'fav_hist_n_tournaments', 'und_hist_n_tournaments',
    'fav_hist_pso_win_rate_shrunk', 'und_hist_pso_win_rate_shrunk',
    'hist_win_rate_diff', 'hist_draw_rate_diff', 'elo_gap', 'abs_elo_gap',
]
TEAM_SQUAD_COLS = [
    'fav_squad_age_mean', 'und_squad_age_mean',
    'fav_squad_prior_wc_mean', 'und_squad_prior_wc_mean',
    'fav_squad_share_any_prior_wc', 'und_squad_share_any_prior_wc',
    'fav_squad_jaccard_vs_prev_wc', 'und_squad_jaccard_vs_prev_wc',
]
TEAM_COACH_COLS = [
    'fav_manager_local', 'und_manager_local',
    'fav_mgr_n_prior_wc', 'und_mgr_n_prior_wc',
    'fav_mgr_hist_win_rate_shrunk', 'und_mgr_hist_win_rate_shrunk',
    'fav_mgr_consecutive_wc_with_team', 'und_mgr_consecutive_wc_with_team',
]
TEAM_RANK_COLS = [
    'fav_elo_rating', 'und_elo_rating',
    'fav_fifa_rank', 'und_fifa_rank',
    'fav_fifa_points', 'und_fifa_points',
]
MATCH_COLS = [
    'fav_is_home', 'group_stage', 'knockout_stage',
    'feat_home_is_host', 'feat_away_is_host', 'feat_same_confederation',
    'rest_days_diff',
]
PLAYER_ONLY_COLS = [
    c for c in df_fav.columns
    if c.startswith('fav_pl_') or c.startswith('und_pl_') or c.startswith('fav_sq_') or c.startswith('und_sq_')
]
DIFF_COLS = [c for c in df_fav.columns if c.startswith('diff_')]

TEAM_ONLY_FEATURES = [
    c for c in (TEAM_HIST_COLS + TEAM_SQUAD_COLS + TEAM_COACH_COLS + TEAM_RANK_COLS + MATCH_COLS)
    if c in df_fav.columns and c != 'abs_elo_gap'
]
PLAYER_ENRICHED_FEATURES = [
    c for c in list(dict.fromkeys(TEAM_ONLY_FEATURES + PLAYER_ONLY_COLS))
    if c in df_fav.columns
]
PLAYER_ENRICHED_WITH_DIFFS = [
    c for c in list(dict.fromkeys(TEAM_ONLY_FEATURES + PLAYER_ONLY_COLS + DIFF_COLS))
    if c in df_fav.columns
]

feature_summary = pd.DataFrame([
    {
        'feature_set': 'TEAM_ONLY_FEATURES',
        'n_features': len(TEAM_ONLY_FEATURES),
        'notes': 'Adjusted team-only set used by both baseline and team-history models',
    },
    {
        'feature_set': 'PLAYER_ENRICHED_FEATURES',
        'n_features': len(PLAYER_ENRICHED_FEATURES),
        'notes': 'Final report-aligned player set (no diff_* player features)',
    },
    {
        'feature_set': 'PLAYER_ENRICHED_WITH_DIFFS',
        'n_features': len(PLAYER_ENRICHED_WITH_DIFFS),
        'notes': 'Exploratory wider player set from 03; not used in the final report',
    },
])
display(feature_summary)


## C. Define the final locked models

The final report uses three systems:

- **Baseline**: adjusted team-only features, trained on 2006-2014, random forest
- **Team-history**: adjusted team-only features, trained on 1998-2014, random forest
- **Player-enriched**: adjusted team features plus favorite/underdog player+squad columns, trained on 2006-2014, XGBoost with the final report-aligned parameters below


In [ ]:
FINAL_CONFIGS = {
    'baseline': {
        'display_name': 'Baseline model',
        'training_window': '2006-2014',
        'train_min_year': 2006,
        'main_signals': 'Team-level only',
        'features': TEAM_ONLY_FEATURES,
        'pipeline': Pipeline([
            ('imp', SimpleImputer()),
            ('clf', RandomForestClassifier(
                n_estimators=300,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),
        'model_family': 'RandomForestClassifier',
        'params': {
            'n_estimators': 300,
            'class_weight': 'balanced',
            'random_state': SEED,
            'n_jobs': -1,
        },
    },
    'team_history': {
        'display_name': 'Team-history model',
        'training_window': '1998-2014',
        'train_min_year': 1998,
        'main_signals': 'Team-level only',
        'features': TEAM_ONLY_FEATURES,
        'pipeline': Pipeline([
            ('imp', SimpleImputer()),
            ('clf', RandomForestClassifier(
                n_estimators=300,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),
        'model_family': 'RandomForestClassifier',
        'params': {
            'n_estimators': 300,
            'class_weight': 'balanced',
            'random_state': SEED,
            'n_jobs': -1,
        },
    },
    'player_enriched': {
        'display_name': 'Player-enriched model',
        'training_window': '2006-2014',
        'train_min_year': 2006,
        'main_signals': 'Team + player/squad',
        'features': PLAYER_ENRICHED_FEATURES,
        'pipeline': Pipeline([
            ('imp', SimpleImputer()),
            ('clf', xgb.XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=SEED,
                verbosity=0,
                n_jobs=-1,
            )),
        ]),
        'model_family': 'XGBClassifier',
        'params': {
            'n_estimators': 300,
            'max_depth': 4,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'use_label_encoder': False,
            'eval_metric': 'mlogloss',
            'random_state': SEED,
            'verbosity': 0,
            'n_jobs': -1,
        },
    },
}

spec_rows = []
for key, cfg in FINAL_CONFIGS.items():
    spec_rows.append({
        'model_key': key,
        'display_name': cfg['display_name'],
        'training_window': cfg['training_window'],
        'main_signals': cfg['main_signals'],
        'model_family': cfg['model_family'],
        'n_features': len(cfg['features']),
        'param_summary': json.dumps(cfg['params'], sort_keys=True),
    })

display(pd.DataFrame(spec_rows))


## D. Helper functions

The report uses:

- **LOTO cross-validation** for the training-era tournaments
- **2018 + 2022 holdout** for final evaluation
- the **same holdout predictions** for both Table 1 and the stage-wise comparison


In [ ]:
def loto_cv_summary(df_train, feature_cols, pipeline):
    years = sorted(df_train['year'].unique())
    accs, f1s, aucs = [], [], []

    for test_year in years:
        tr = df_train[df_train['year'] != test_year]
        te = df_train[df_train['year'] == test_year]
        model = clone(pipeline)
        model.fit(tr[feature_cols], tr['y'])
        pred = model.predict(te[feature_cols])
        prob = model.predict_proba(te[feature_cols])
        yb = label_binarize(te['y'], classes=[0, 1, 2])

        accs.append(accuracy_score(te['y'], pred))
        f1s.append(f1_score(te['y'], pred, average='macro', zero_division=0))
        aucs.append(roc_auc_score(yb, prob, average='macro', multi_class='ovr'))

    return {
        'cv_acc': float(np.mean(accs)),
        'cv_acc_std': float(np.std(accs)),
        'cv_f1_macro': float(np.mean(f1s)),
        'cv_f1_std': float(np.std(f1s)),
        'cv_auc_macro': float(np.mean(aucs)),
        'cv_auc_std': float(np.std(aucs)),
        'cv_years': years,
    }


def fit_on_full_training(df, feature_cols, pipeline, train_min_year):
    train = df[(df['year'] >= train_min_year) & (~df['year'].isin(HOLDOUT_YEARS))].copy()
    holdout = df[df['year'].isin(HOLDOUT_YEARS)].copy()

    model = clone(pipeline)
    model.fit(train[feature_cols], train['y'])

    pred = model.predict(holdout[feature_cols])
    prob = model.predict_proba(holdout[feature_cols])
    holdout_pred = holdout[['match_id', 'tournament_id', 'year', 'y', 'group_stage', 'knockout_stage', 'fav_team_id', 'und_team_id']].copy()
    holdout_pred['pred'] = pred
    holdout_pred['prob_fav_loses'] = prob[:, 0]
    holdout_pred['prob_draw'] = prob[:, 1]
    holdout_pred['prob_fav_wins'] = prob[:, 2]

    metrics = {
        'holdout_acc': float(accuracy_score(holdout_pred['y'], holdout_pred['pred'])),
        'holdout_f1_macro': float(f1_score(holdout_pred['y'], holdout_pred['pred'], average='macro', zero_division=0)),
        'overall_confusion': confusion_matrix(holdout_pred['y'], holdout_pred['pred'], labels=[0, 1, 2]).tolist(),
    }
    return model, holdout_pred, metrics


def stage_metrics(pred_df):
    rows = []
    stage_masks = {
        'group_stage': pred_df['group_stage'].fillna(0).astype(bool),
        'knockout_stage': pred_df['knockout_stage'].fillna(0).astype(bool),
        'overall': pd.Series([True] * len(pred_df), index=pred_df.index),
    }
    for stage, mask in stage_masks.items():
        sub = pred_df[mask]
        rows.append({
            'stage': stage,
            'n_matches': int(len(sub)),
            'accuracy': float(accuracy_score(sub['y'], sub['pred'])),
            'macro_f1': float(f1_score(sub['y'], sub['pred'], average='macro', zero_division=0)),
            'confusion': confusion_matrix(sub['y'], sub['pred'], labels=[0, 1, 2]).tolist(),
        })
    return pd.DataFrame(rows)


## E. Reproduce the final report metrics

This section computes the final report summary table directly from the locked models above.


In [ ]:
results = {}
report_rows = []

for model_key, cfg in FINAL_CONFIGS.items():
    train_df = df_fav[(df_fav['year'] >= cfg['train_min_year']) & (~df_fav['year'].isin(HOLDOUT_YEARS))].copy()
    cv = loto_cv_summary(train_df, cfg['features'], cfg['pipeline'])
    fitted_model, holdout_pred, holdout_metrics = fit_on_full_training(
        df_fav,
        cfg['features'],
        cfg['pipeline'],
        cfg['train_min_year'],
    )
    stage_df = stage_metrics(holdout_pred)

    results[model_key] = {
        'config': cfg,
        'cv': cv,
        'fitted_model': fitted_model,
        'holdout_pred': holdout_pred,
        'holdout_metrics': holdout_metrics,
        'stage_df': stage_df,
    }

    report_rows.append({
        'model_key': model_key,
        'display_name': cfg['display_name'],
        'training_window': cfg['training_window'],
        'main_signals': cfg['main_signals'],
        'model_family': cfg['model_family'],
        'n_features': len(cfg['features']),
        'cv_accuracy': cv['cv_acc'],
        'holdout_accuracy': holdout_metrics['holdout_acc'],
        'holdout_macro_f1': holdout_metrics['holdout_f1_macro'],
    })

table1_df = pd.DataFrame(report_rows).set_index('model_key')
display(
    table1_df[[
        'display_name', 'training_window', 'main_signals', 'model_family', 'n_features',
        'cv_accuracy', 'holdout_accuracy', 'holdout_macro_f1',
    ]].style.format({
        'cv_accuracy': '{:.3f}',
        'holdout_accuracy': '{:.3f}',
        'holdout_macro_f1': '{:.3f}',
    })
)


In [ ]:
EXPECTED_TABLE1 = {
    'baseline': {'cv_accuracy': 0.682292, 'holdout_accuracy': 0.656250, 'holdout_macro_f1': 0.265403},
    'team_history': {'cv_accuracy': 0.662500, 'holdout_accuracy': 0.648438, 'holdout_macro_f1': 0.291866},
    'player_enriched': {'cv_accuracy': 0.682292, 'holdout_accuracy': 0.656250, 'holdout_macro_f1': 0.340019},
}
EXPECTED_OVERALL_CONFUSIONS = {
    'baseline': [[0, 1, 23], [0, 0, 19], [0, 1, 84]],
    'team_history': [[0, 0, 24], [0, 1, 18], [1, 2, 82]],
    'player_enriched': [[2, 1, 21], [0, 1, 18], [2, 2, 81]],
}

for model_key, expected in EXPECTED_TABLE1.items():
    row = table1_df.loc[model_key]
    for metric_name, expected_value in expected.items():
        actual = float(row[metric_name])
        assert abs(actual - expected_value) < 1e-6, f'{model_key} {metric_name}: {actual} != {expected_value}'
    actual_cm = results[model_key]['holdout_metrics']['overall_confusion']
    assert actual_cm == EXPECTED_OVERALL_CONFUSIONS[model_key], f'{model_key} confusion mismatch: {actual_cm}'

print('Table 1 lock checks passed.')


## F. Stage-wise summary using the same holdout predictions

This reproduces the stage narrative used in the report. No separate refit is used here; the stage splits come from the same locked holdout predictions used in Table 1.


In [ ]:
stage_rows = []
for model_key in ['team_history', 'player_enriched']:
    stage_df = results[model_key]['stage_df'].copy()
    stage_df['model_key'] = model_key
    stage_df['display_name'] = results[model_key]['config']['display_name']
    stage_rows.append(stage_df)

stage_long = pd.concat(stage_rows, ignore_index=True)
stage_table = stage_long.pivot(index='stage', columns='model_key', values=['accuracy', 'macro_f1', 'n_matches'])
display(stage_table.round(6))

stage_compact = pd.DataFrame({
    'n_matches': stage_table[('n_matches', 'team_history')].astype(int),
    'team_history_acc': stage_table[('accuracy', 'team_history')],
    'player_enriched_acc': stage_table[('accuracy', 'player_enriched')],
    'team_history_f1': stage_table[('macro_f1', 'team_history')],
    'player_enriched_f1': stage_table[('macro_f1', 'player_enriched')],
})
display(stage_compact.style.format({
    'team_history_acc': '{:.3f}',
    'player_enriched_acc': '{:.3f}',
    'team_history_f1': '{:.3f}',
    'player_enriched_f1': '{:.3f}',
}))


In [ ]:
EXPECTED_STAGE_CONFUSIONS = {
    ('team_history', 'group_stage'): [[0, 0, 15], [0, 1, 18], [1, 2, 59]],
    ('team_history', 'knockout_stage'): [[0, 0, 9], [0, 0, 0], [0, 0, 23]],
    ('player_enriched', 'group_stage'): [[0, 1, 14], [0, 1, 18], [2, 2, 58]],
    ('player_enriched', 'knockout_stage'): [[2, 0, 7], [0, 0, 0], [0, 0, 23]],
}
EXPECTED_STAGE_METRICS = {
    ('team_history', 'group_stage'): {'accuracy': 0.625000, 'macro_f1': 0.285714},
    ('team_history', 'knockout_stage'): {'accuracy': 0.718750, 'macro_f1': 0.418182},
    ('team_history', 'overall'): {'accuracy': 0.648438, 'macro_f1': 0.291866},
    ('player_enriched', 'group_stage'): {'accuracy': 0.614583, 'macro_f1': 0.283371},
    ('player_enriched', 'knockout_stage'): {'accuracy': 0.781250, 'macro_f1': 0.615780},
    ('player_enriched', 'overall'): {'accuracy': 0.656250, 'macro_f1': 0.340019},
}

for model_key in ['team_history', 'player_enriched']:
    stage_df = results[model_key]['stage_df']
    for _, row in stage_df.iterrows():
        stage = row['stage']
        exp = EXPECTED_STAGE_METRICS[(model_key, stage)]
        assert abs(float(row['accuracy']) - exp['accuracy']) < 1e-6, (model_key, stage, row['accuracy'], exp['accuracy'])
        assert abs(float(row['macro_f1']) - exp['macro_f1']) < 1e-6, (model_key, stage, row['macro_f1'], exp['macro_f1'])
        if stage in ('group_stage', 'knockout_stage'):
            assert row['confusion'] == EXPECTED_STAGE_CONFUSIONS[(model_key, stage)], (model_key, stage, row['confusion'])

print('Stage-wise lock checks passed.')


## G. Final confusion matrices

These are the exact matrices used by the final report logic.


In [ ]:
def cm_df(cm):
    return pd.DataFrame(cm, index=CLASS_LABELS, columns=CLASS_LABELS)

print('Overall confusion matrices')
for model_key in ['baseline', 'team_history', 'player_enriched']:
    print('\n' + FINAL_CONFIGS[model_key]['display_name'])
    display(cm_df(results[model_key]['holdout_metrics']['overall_confusion']))

print('\nStage-wise confusion matrices')
for model_key in ['team_history', 'player_enriched']:
    print(f'\n{FINAL_CONFIGS[model_key]["display_name"]}')
    for _, row in results[model_key]['stage_df'].iterrows():
        if row['stage'] == 'overall':
            continue
        print('  ', row['stage'])
        display(cm_df(row['confusion']))


## H. Export locked artifacts

Running this cell writes lightweight lock files so the final report numbers are not trapped inside notebook outputs.


In [ ]:
LOCK_DIR.mkdir(parents=True, exist_ok=True)

spec_export = {
    key: {
        'display_name': cfg['display_name'],
        'training_window': cfg['training_window'],
        'train_min_year': cfg['train_min_year'],
        'main_signals': cfg['main_signals'],
        'model_family': cfg['model_family'],
        'n_features': len(cfg['features']),
        'feature_list': cfg['features'],
        'params': cfg['params'],
    }
    for key, cfg in FINAL_CONFIGS.items()
}

(LOCK_DIR / 'final_model_specs.json').write_text(json.dumps(spec_export, indent=2))
table1_df.to_csv(LOCK_DIR / 'table1_summary.csv', index=True)
stage_long.to_csv(LOCK_DIR / 'stage_summary_long.csv', index=False)
stage_compact.to_csv(LOCK_DIR / 'stage_summary_compact.csv', index=True)

conf_export = {
    'overall': {key: results[key]['holdout_metrics']['overall_confusion'] for key in results},
    'stage': {
        key: {
            row['stage']: row['confusion']
            for _, row in results[key]['stage_df'].iterrows()
        }
        for key in ['team_history', 'player_enriched']
    },
}
(LOCK_DIR / 'confusion_matrices.json').write_text(json.dumps(conf_export, indent=2))

holdout_frames = []
for key, result in results.items():
    pred_df = result['holdout_pred'].copy()
    rename_map = {
        'pred': f'{key}_pred',
        'prob_fav_loses': f'{key}_prob_fav_loses',
        'prob_draw': f'{key}_prob_draw',
        'prob_fav_wins': f'{key}_prob_fav_wins',
    }
    if holdout_frames:
        pred_df = pred_df[['match_id', 'pred', 'prob_fav_loses', 'prob_draw', 'prob_fav_wins']].rename(columns=rename_map)
    else:
        pred_df = pred_df.rename(columns=rename_map)
    holdout_frames.append(pred_df)

holdout_locked = holdout_frames[0]
for extra in holdout_frames[1:]:
    holdout_locked = holdout_locked.merge(extra, on='match_id', how='left')

holdout_locked.to_parquet(LOCK_DIR / 'holdout_predictions.parquet', index=False)

print('Locked artifacts written to:', LOCK_DIR)
print(sorted(p.name for p in LOCK_DIR.iterdir()))


## I. Final note

From this point onward, this notebook should be treated as the **report lock**. If future edits are needed, update them here first and then propagate the numbers or figures outward.
